# Demostración del genético de HortyFru

Este cuaderno ejecuta el mismo módulo Python del backend. El nodo 0 representa el almacén y todas las soluciones regresan a él. Se comparan costos de la misma matriz. Las matrices sintéticas y TSPLIB no son mediciones de tráfico en La Paz.

En Colab, subir y extraer una copia del proyecto o clonar el repositorio autorizado. No hace falta iniciar Django ni copiar archivos .env. Seleccionar el entorno CPU.

In [ ]:
from pathlib import Path

# Ajustar a la carpeta que contiene Backend/ e investigacion/.
raiz = Path.cwd()
assert (raiz / "Backend/despachos/optimizacion/genetico.py").exists(), "Selecciona la raíz del proyecto"


In [ ]:
%pip install -r {raiz / "investigacion/rutas/requirements.txt"}


In [ ]:
import sys
sys.path.insert(0, str(raiz / "Backend"))
sys.path.insert(0, str(raiz / "investigacion/rutas"))
from despachos.optimizacion.genetico import optimizar_recorrido, validar_matriz, costo_recorrido, ParametrosGenetico
from evaluar_genetico import ejecutar_experimentos, calibrar


## 1. Matriz de prueba y óptimo independiente

Los valores siguientes son costos sintéticos expresados en segundos. La matriz es asimétrica. Enumerar todos los órdenes es viable en este ejemplo pequeño y permite verificar la referencia sin depender del genético.

In [ ]:
import itertools
import json
matrices = raiz / "investigacion/rutas/matrices"
caso = json.loads((matrices / "01_pequena_asimetrica.json").read_text(encoding="utf-8"))
matriz = validar_matriz(caso["matriz"])
orden_exacto = min(itertools.permutations(range(1, len(matriz))), key=lambda orden: costo_recorrido(matriz, orden))
print("Ruta exacta:", [0, *orden_exacto, 0])
print("Costo exacto:", costo_recorrido(matriz, orden_exacto))


## 2. Ejecutar e inspeccionar el mismo algoritmo

La élite se conserva explícitamente en cada generación. El generador aleatorio es local a cada ejecución. El archivo no depende de la base de datos ni de la API.

In [ ]:
resultado = optimizar_recorrido(caso["matriz"], semilla=42)
print("Ruta propuesta:", [0, *resultado["orden"], 0])
print("Costo:", resultado["costo"], resultado["unidad"])
print("Parámetros:", resultado["parametros"])
assert sorted(resultado["orden"]) == list(range(1, len(matriz)))
assert resultado == optimizar_recorrido(caso["matriz"], semilla=42)


## 3. Calibración separada

Este paso utiliza matrices y semillas de generación diferentes del conjunto final. Registra las configuraciones comparadas. Ejecutarlo antes de evaluar y documentar los parámetros elegidos; no ajustarlos mirando los casos de evaluación.

In [ ]:
salida = raiz / "investigacion/rutas/resultados_cuaderno"
# Opcional: repetir la calibración documentada.
# calibracion = calibrar(salida)
print(ParametrosGenetico())


## 4. Treinta ejecuciones por matriz

Se comparan orden actual, vecino más cercano, genético y OR-Tools. OR-Tools es una referencia heurística, no un certificado de óptimo. Su presupuesto de búsqueda se iguala aproximadamente al tiempo medido del genético y se registra el tiempo total. Los costos TSPLIB conservan sus unidades originales. Para medir tiempos comparables, ejecutar los métodos secuencialmente en el mismo equipo.

In [ ]:
reporte = ejecutar_experimentos(matrices, salida, repeticiones=30)
reporte["resumen"]


## 5. Revisar evidencia

Se guardan ejecuciones.csv, resumen.json, historiales.json y la gráfica de convergencia PNG/PDF. El resumen registra versiones, parámetros y huellas de las matrices. El costo estimado de viaje se distingue del tiempo real de cálculo. Las conclusiones se limitan a las instancias evaluadas; una curva de convergencia no certifica optimalidad.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(salida / "convergencia.png")))
print((salida / "resumen.json").read_text(encoding="utf-8"))


## Referencias

- DEAP, estructuras y operadores: https://deap.readthedocs.io/en/master/api/tools.html
- Ejemplo TSP de DEAP: https://github.com/DEAP/deap/blob/master/examples/ga/tsp.py
- TSPLIB y referencias: https://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/
- OSRM Table/Route: https://project-osrm.org/docs/v26.6.1/http

El cruce OX de nuestra implementación sigue el operador documentado por DEAP, adaptado a un generador local y a índices 1..n. Las decisiones de evaluación y su integración se explican en README.md.